# 03 — Positive Inventory Deduplication

**FYP:** Developing a Semi-Dynamic Landslide Susceptibility Decision-Support Dashboard Using Machine Learning for Malaysia

## Purpose

Stage 2 successfully resolved all 52 JMG occurrence records into WGS84 coordinates.

This notebook combines:

- **NASA COOLR Malaysia:** 236 raw occurrence records
- **JMG government inventory:** 52 resolved occurrence records

and determines the **defensible final positive-event inventory** before pseudo-absence generation.

## Deduplication principles

This notebook is deliberately conservative:

1. A repeated ID alone is **not enough** to delete an event.
2. A repeated coordinate alone is **not enough** to delete an event if event dates differ.
3. A cross-source spatially close pair is **not enough** to declare a duplicate.
4. Strong duplicates require multiple consistent signals, especially:
   - same/near-identical location,
   - same or very close event date,
   - same descriptive information where available.
5. Records are never silently removed.
6. Every exclusion receives a documented duplicate-group ID and reason.

## Expected current evidence

From Stage 2:

- JMG = 52 resolved points.
- 52/52 are inside Malaysia.
- Six JMG rows were flagged for duplicate review, but repeated IDs/locations occur with differing dates or locations and should not automatically be removed.
- COOLR contains one exact-coordinate/date/title pair at Labuan that is a strong internal duplicate candidate.

The notebook will verify all of this again from the actual files.


In [1]:
# OPTIONAL: run only if packages are missing.
# !pip install -q geopandas pandas numpy matplotlib shapely

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)

print("pandas:", pd.__version__)
print("geopandas:", gpd.__version__)


pandas: 3.0.2
geopandas: 1.1.4


## 1. Project paths

Change **only `PROJECT_ROOT`**.

Expected inputs:

```text
Landslide_FYP_Data/
└── 01_Inventory/
    └── NASA_COOLR/
        └── coolr_malaysia_candidates.geojson

Landslide_FYP_Development/
└── 02_cleaned_inventory/
    └── jmg_inventory_wgs84_resolved.geojson
```

Outputs will be written into:

```text
Landslide_FYP_Development/
├── 01_preprocessing/
│   └── positive_inventory_deduplication_outputs/
└── 02_cleaned_inventory/
    ├── malaysia_positive_inventory_final.geojson
    └── malaysia_positive_inventory_final.csv
```


In [2]:
# ============================================================
# CHANGE ONLY THIS LINE
# ============================================================
PROJECT_ROOT = Path(r"C:\Landslide_FYP")

RAW_DIR = PROJECT_ROOT / "Landslide_FYP_Data"
DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"

COOLR_PATH = (
    RAW_DIR
    / "01_Inventory"
    / "NASA_COOLR"
    / "coolr_malaysia_candidates.geojson"
)

JMG_PATH = (
    DEV_DIR
    / "02_cleaned_inventory"
    / "jmg_inventory_wgs84_resolved.geojson"
)

BOUNDARY_PATH = (
    RAW_DIR
    / "02_Boundary"
    / "mys_admin0.geojson"
)

OUTPUT_DIR = (
    DEV_DIR
    / "01_preprocessing"
    / "positive_inventory_deduplication_outputs"
)

CLEAN_DIR = DEV_DIR / "02_cleaned_inventory"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

for name, path in {
    "NASA COOLR": COOLR_PATH,
    "Resolved JMG": JMG_PATH,
    "Malaysia boundary": BOUNDARY_PATH,
}.items():
    print(f"{name}: {path}")
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file:\\n{path}"
        )

print("\\nAll required Stage-3 inputs were found.")


NASA COOLR: C:\Landslide_FYP\Landslide_FYP_Data\01_Inventory\NASA_COOLR\coolr_malaysia_candidates.geojson
Resolved JMG: C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\jmg_inventory_wgs84_resolved.geojson
Malaysia boundary: C:\Landslide_FYP\Landslide_FYP_Data\02_Boundary\mys_admin0.geojson
\nAll required Stage-3 inputs were found.


## 2. Load and validate inputs

In [3]:
coolr = gpd.read_file(COOLR_PATH)
jmg = gpd.read_file(JMG_PATH)
malaysia = gpd.read_file(BOUNDARY_PATH)

print("COOLR:", coolr.shape, coolr.crs)
print("JMG:", jmg.shape, jmg.crs)
print("Boundary:", malaysia.shape, malaysia.crs)

if len(coolr) != 236:
    print(
        f"WARNING: expected 236 COOLR rows from the current workflow, "
        f"but found {len(coolr)}."
    )

if len(jmg) != 52:
    raise ValueError(
        f"Expected 52 resolved JMG rows, found {len(jmg)}."
    )

if coolr.crs is None or jmg.crs is None:
    raise ValueError("Input CRS is missing.")

coolr = coolr.to_crs("EPSG:4326")
jmg = jmg.to_crs("EPSG:4326")

print("\nInputs standardized to EPSG:4326.")


COOLR: (236, 31) EPSG:4326
JMG: (52, 62) EPSG:4326
Boundary: (1, 18) EPSG:4326

Inputs standardized to EPSG:4326.


## 3. Standardise audit fields

Original source attributes remain unchanged.

New fields are added only for deduplication and final inventory management.


In [4]:
# ------------------------------------------------------------
# COOLR
# ------------------------------------------------------------
coolr_work = coolr.copy()

coolr_work["source_dataset"] = "NASA_COOLR"
coolr_work["source_record_id"] = (
    "COOLR_" + coolr_work["event_id"].astype("string")
)

coolr_work["event_date_standard"] = pd.to_datetime(
    coolr_work["event_date"],
    unit="ms",
    errors="coerce",
    utc=True
).dt.tz_localize(None)

coolr_work["final_longitude"] = coolr_work.geometry.x
coolr_work["final_latitude"] = coolr_work.geometry.y

# ------------------------------------------------------------
# JMG
# ------------------------------------------------------------
jmg_work = jmg.copy()

jmg_work["source_dataset"] = "JMG"
jmg_work["source_record_id"] = jmg_work["jmg_resolved_id"].astype("string")

jmg_work["event_date_standard"] = pd.to_datetime(
    jmg_work["source_date_candidate"],
    errors="coerce"
)

jmg_work["final_longitude"] = jmg_work.geometry.x
jmg_work["final_latitude"] = jmg_work.geometry.y

print("COOLR standard dates:", coolr_work["event_date_standard"].notna().sum())
print("JMG standard dates:", jmg_work["event_date_standard"].notna().sum())


COOLR standard dates: 236
JMG standard dates: 47


## 4. Text normalisation helper

Text similarity is used only as supporting evidence.  
It never creates or fills missing source information.


In [5]:
def normalize_text(value):
    if pd.isna(value):
        return ""
    value = str(value).lower().strip()
    value = re.sub(r"[^a-z0-9]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value

for col in ["event_title", "location_description"]:
    if col in coolr_work.columns:
        coolr_work[f"_norm_{col}"] = coolr_work[col].apply(normalize_text)

for col in ["Nama_Kampu", "Nama_Jalan", "Daerah", "Negeri"]:
    if col in jmg_work.columns:
        jmg_work[f"_norm_{col}"] = jmg_work[col].apply(normalize_text)


## 5. Within-COOLR duplicate screening

Strong automatic duplicate evidence requires:

- exact same latitude/longitude to 6 decimal places,
- same event date,
- and same normalised title or same normalised location description.

A row is retained as the representative record; additional rows in the same strong duplicate group are marked `EXCLUDE_DUPLICATE`.

Rows with only the same coordinate but different dates remain retained.


In [6]:
coolr_work["_lat6"] = coolr_work["final_latitude"].round(6)
coolr_work["_lon6"] = coolr_work["final_longitude"].round(6)
coolr_work["_date_day"] = coolr_work["event_date_standard"].dt.normalize()

coolr_work["dedup_status"] = "KEEP"
coolr_work["duplicate_group"] = pd.NA
coolr_work["duplicate_reason"] = pd.NA
coolr_work["representative_record_id"] = pd.NA

coolr_exact_coord_groups = (
    coolr_work[
        coolr_work.duplicated(["_lat6", "_lon6"], keep=False)
    ]
    .sort_values(["_lat6", "_lon6", "_date_day", "event_id"])
)

print("COOLR rows sharing exact rounded coordinates:", len(coolr_exact_coord_groups))

display(
    coolr_exact_coord_groups[
        [
            "source_record_id",
            "event_id",
            "event_date_standard",
            "event_title",
            "location_description",
            "location_accuracy",
            "final_latitude",
            "final_longitude",
        ]
    ]
)


COOLR rows sharing exact rounded coordinates: 2


,source_record_id,event_id,event_date_standard,event_title,location_description,location_accuracy,final_latitude,final_longitude
11,COOLR_9082,9082,2015-12-19,Labuan,Labuan,25km,5.3149,115.2203
0,COOLR_9083,9083,2015-12-19,Labuan,Labuan,25km,5.3149,115.2203


In [7]:
coolr_group_counter = 1
coolr_auto_duplicate_rows = []

for (_, _), grp in coolr_work.groupby(["_lat6", "_lon6"], dropna=False):
    if len(grp) < 2:
        continue

    # Split same-location records by event day.
    for date_value, date_grp in grp.groupby("_date_day", dropna=False):
        if len(date_grp) < 2 or pd.isna(date_value):
            continue

        title_values = date_grp["_norm_event_title"].fillna("")
        loc_values = date_grp["_norm_location_description"].fillna("")

        same_title = (
            len(set(v for v in title_values if v)) == 1
            and (title_values != "").all()
        )

        same_location = (
            len(set(v for v in loc_values if v)) == 1
            and (loc_values != "").all()
        )

        if not (same_title or same_location):
            continue

        # Stable representative: lowest event_id, then source_record_id.
        ordered = date_grp.sort_values(
            ["event_id", "source_record_id"],
            na_position="last"
        )

        representative_idx = ordered.index[0]
        representative_id = ordered.iloc[0]["source_record_id"]
        group_id = f"COOLR_DUP_{coolr_group_counter:03d}"

        coolr_work.loc[ordered.index, "duplicate_group"] = group_id
        coolr_work.loc[ordered.index, "representative_record_id"] = representative_id
        coolr_work.loc[ordered.index, "duplicate_reason"] = (
            "Same coordinate + same event date + matching title/location"
        )

        # Keep first, exclude subsequent records.
        exclude_indices = ordered.index[1:]
        coolr_work.loc[exclude_indices, "dedup_status"] = "EXCLUDE_DUPLICATE"

        for idx in ordered.index:
            coolr_auto_duplicate_rows.append(idx)

        coolr_group_counter += 1

coolr_auto_review = coolr_work.loc[
    sorted(set(coolr_auto_duplicate_rows)),
    [
        "source_record_id",
        "event_id",
        "event_date_standard",
        "event_title",
        "location_description",
        "location_accuracy",
        "final_latitude",
        "final_longitude",
        "dedup_status",
        "duplicate_group",
        "representative_record_id",
        "duplicate_reason",
    ]
].copy()

print("Strong COOLR duplicate groups:", coolr_group_counter - 1)
print(
    "COOLR rows marked EXCLUDE_DUPLICATE:",
    (coolr_work["dedup_status"] == "EXCLUDE_DUPLICATE").sum()
)

display(coolr_auto_review)

coolr_auto_review.to_csv(
    OUTPUT_DIR / "coolr_strong_duplicate_review.csv",
    index=False
)


Strong COOLR duplicate groups: 1
COOLR rows marked EXCLUDE_DUPLICATE: 1


,source_record_id,event_id,event_date_standard,event_title,location_description,location_accuracy,final_latitude,final_longitude,dedup_status,duplicate_group,representative_record_id,duplicate_reason
0,COOLR_9083,9083,2015-12-19,Labuan,Labuan,25km,5.3149,115.2203,EXCLUDE_DUPLICATE,COOLR_DUP_001,COOLR_9082,Same coordinate + same event date + matching t...
11,COOLR_9082,9082,2015-12-19,Labuan,Labuan,25km,5.3149,115.2203,KEEP,COOLR_DUP_001,COOLR_9082,Same coordinate + same event date + matching t...


## 6. Within-JMG duplicate review

Stage 2 found repeated IDs and one repeated exact location.

Here the rule is stricter:

A JMG record is **not automatically removed** merely because:

- its source event ID repeats, or
- its exact coordinate repeats.

For automatic exclusion, records would need the **same exact location and same valid event date**, plus matching identifying evidence.

This protects legitimate recurrent landslides at the same location.


In [8]:
jmg_work["_lat6"] = jmg_work["final_latitude"].round(6)
jmg_work["_lon6"] = jmg_work["final_longitude"].round(6)
jmg_work["_date_day"] = jmg_work["event_date_standard"].dt.normalize()

jmg_work["dedup_status"] = "KEEP"
jmg_work["duplicate_group"] = pd.NA
jmg_work["duplicate_reason"] = pd.NA
jmg_work["representative_record_id"] = pd.NA

jmg_internal_flags = (
    jmg_work["duplicate_source_event_id"].fillna(False)
    | jmg_work["duplicate_exact_projected_xy"].fillna(False)
    | jmg_work["duplicate_resolved_wgs84"].fillna(False)
)

jmg_review = jmg_work.loc[
    jmg_internal_flags,
    [
        "source_record_id",
        "jmg_source_event_id",
        "SOURCE_FILE",
        "event_date_standard",
        "Nama_Kampu",
        "Nama_Jalan",
        "Daerah",
        "Negeri",
        "final_latitude",
        "final_longitude",
        "duplicate_source_event_id",
        "duplicate_exact_projected_xy",
        "duplicate_resolved_wgs84",
    ]
].copy()

print("JMG rows requiring within-source review:", len(jmg_review))
display(jmg_review)

jmg_review.to_csv(
    OUTPUT_DIR / "jmg_within_source_duplicate_review.csv",
    index=False
)


JMG rows requiring within-source review: 6


,source_record_id,jmg_source_event_id,SOURCE_FILE,event_date_standard,Nama_Kampu,Nama_Jalan,Daerah,Negeri,final_latitude,final_longitude,duplicate_source_event_id,duplicate_exact_projected_xy,duplicate_resolved_wgs84
36,JMG_037,AY44201_LS_05,ch_TANAH_RUNTUH_SEMENANJUNG_.shp,2021-12-19,Rumah Istirehat KWSP,NaN,Cameron Highlands,Pahang,4.491970,101.379137,True,False,False
37,JMG_038,AY44201_LS_05,ch_TANAH_RUNTUH_SEMENANJUNG_.shp,2021-12-19,PRIMA,NaN,Cameron Highlands,Pahang,4.476578,101.376696,True,False,False
41,JMG_042,AY44201_M_01_SF_01,ch_TANAH_RUNTUH_SEMENANJUNG_.shp,2021-05-29,Taman Gambang Damai,NaN,Cameron Highlands,Pahang,4.491990,101.385995,False,True,True
42,JMG_043,AY44201_M_02_SF_01,ch_TANAH_RUNTUH_SEMENANJUNG_.shp,2021-01-06,Jalan Turun Bukit Fraser (KM 6.4),NaN,Cameron Highlands,Pahang,4.491990,101.385995,False,True,True
44,JMG_045,FH2101_M_01_SF_01,pp_TANAH_RUNTUH_SEMENANJUNG_.shp,2021-10-30,"664G, Pepper Estate",NaN,Timor Laut,Pulau Pinang,5.454412,100.297046,True,False,False
45,JMG_046,FH2101_M_01_SF_01,pp_TANAH_RUNTUH_SEMENANJUNG_.shp,2021-09-28,NaN,Jalan Lembah Permai,Timor Laut,Pulau Pinang,5.455675,100.295453,True,False,False


In [9]:
# Search for truly strong JMG same-event duplicates.
jmg_strong_duplicate_groups = []
jmg_group_counter = 1

for (_, _, date_value), grp in jmg_work.groupby(
    ["_lat6", "_lon6", "_date_day"],
    dropna=False
):
    if len(grp) < 2:
        continue

    # Do not infer duplicate events when date is missing.
    if pd.isna(date_value):
        continue

    ids = grp["jmg_source_event_id"].dropna().astype(str).str.strip()
    kampung = grp["_norm_Nama_Kampu"].fillna("") if "_norm_Nama_Kampu" in grp else pd.Series()
    jalan = grp["_norm_Nama_Jalan"].fillna("") if "_norm_Nama_Jalan" in grp else pd.Series()

    same_id = len(ids) == len(grp) and ids.nunique() == 1
    same_kampung = len(kampung) == len(grp) and (kampung != "").all() and kampung.nunique() == 1
    same_jalan = len(jalan) == len(grp) and (jalan != "").all() and jalan.nunique() == 1

    if not (same_id or same_kampung or same_jalan):
        continue

    ordered = grp.sort_values("source_record_id")
    representative_id = ordered.iloc[0]["source_record_id"]
    group_id = f"JMG_DUP_{jmg_group_counter:03d}"

    jmg_work.loc[ordered.index, "duplicate_group"] = group_id
    jmg_work.loc[ordered.index, "representative_record_id"] = representative_id
    jmg_work.loc[ordered.index, "duplicate_reason"] = (
        "Same coordinate + same event date + matching JMG identifying information"
    )
    jmg_work.loc[ordered.index[1:], "dedup_status"] = "EXCLUDE_DUPLICATE"

    jmg_strong_duplicate_groups.append(group_id)
    jmg_group_counter += 1

print("Strong JMG duplicate groups:", len(jmg_strong_duplicate_groups))
print(
    "JMG rows marked EXCLUDE_DUPLICATE:",
    (jmg_work["dedup_status"] == "EXCLUDE_DUPLICATE").sum()
)


Strong JMG duplicate groups: 0
JMG rows marked EXCLUDE_DUPLICATE: 0


## 7. Cross-source JMG ↔ COOLR duplicate candidate generation

Distance alone is not enough because COOLR location accuracy varies (`exact`, `1km`, `5km`, `25km`, etc.).

The notebook therefore generates candidates using a broad **5 km screening radius**, then calculates:

- spatial distance,
- date difference,
- whether dates are within 1, 7, 30, or 90 days,
- COOLR location accuracy,
- JMG district/state,
- COOLR title/location description.

### Conservative strong cross-source duplicate rule

A pair is automatically considered a **strong cross-source duplicate** only when:

- distance ≤ 1 km, **and**
- both dates are available, **and**
- absolute date difference ≤ 7 days.

Anything else remains `REVIEW_ONLY`.

This prevents nearby but historically different events from being merged.


In [10]:
EARTH_RADIUS_KM = 6371.0088

def haversine_km(lat1, lon1, lat2_array, lon2_array):
    lat1 = np.radians(float(lat1))
    lon1 = np.radians(float(lon1))
    lat2 = np.radians(np.asarray(lat2_array, dtype=float))
    lon2 = np.radians(np.asarray(lon2_array, dtype=float))

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(a))

cross_rows = []

coolr_lats = coolr_work["final_latitude"].to_numpy(float)
coolr_lons = coolr_work["final_longitude"].to_numpy(float)

for _, jr in jmg_work.iterrows():
    distances = haversine_km(
        jr["final_latitude"],
        jr["final_longitude"],
        coolr_lats,
        coolr_lons,
    )

    candidate_indices = np.where(distances <= 5.0)[0]

    for pos in candidate_indices:
        cr = coolr_work.iloc[pos]
        distance_km = float(distances[pos])

        if pd.notna(jr["event_date_standard"]) and pd.notna(cr["event_date_standard"]):
            date_difference_days = abs(
                (
                    jr["event_date_standard"].normalize()
                    - cr["event_date_standard"].normalize()
                ).days
            )
        else:
            date_difference_days = np.nan

        strong = (
            distance_km <= 1.0
            and pd.notna(date_difference_days)
            and date_difference_days <= 7
        )

        cross_rows.append({
            "jmg_record_id": jr["source_record_id"],
            "jmg_source_event_id": jr["jmg_source_event_id"],
            "jmg_date": jr["event_date_standard"],
            "jmg_state": jr.get("Negeri"),
            "jmg_district": jr.get("Daerah"),
            "jmg_location_1": jr.get("Nama_Kampu"),
            "jmg_location_2": jr.get("Nama_Jalan"),
            "jmg_latitude": jr["final_latitude"],
            "jmg_longitude": jr["final_longitude"],

            "coolr_record_id": cr["source_record_id"],
            "coolr_event_id": cr["event_id"],
            "coolr_date": cr["event_date_standard"],
            "coolr_title": cr.get("event_title"),
            "coolr_location": cr.get("location_description"),
            "coolr_location_accuracy": cr.get("location_accuracy"),
            "coolr_latitude": cr["final_latitude"],
            "coolr_longitude": cr["final_longitude"],

            "distance_km": round(distance_km, 4),
            "date_difference_days": date_difference_days,
            "date_within_1_day": (
                pd.notna(date_difference_days)
                and date_difference_days <= 1
            ),
            "date_within_7_days": (
                pd.notna(date_difference_days)
                and date_difference_days <= 7
            ),
            "date_within_30_days": (
                pd.notna(date_difference_days)
                and date_difference_days <= 30
            ),
            "date_within_90_days": (
                pd.notna(date_difference_days)
                and date_difference_days <= 90
            ),
            "cross_source_status": (
                "STRONG_DUPLICATE_CANDIDATE"
                if strong
                else "REVIEW_ONLY"
            ),
        })

cross_candidates = pd.DataFrame(cross_rows)

print("Cross-source pairs within 5 km:", len(cross_candidates))

if len(cross_candidates):
    display(
        cross_candidates["cross_source_status"]
        .value_counts(dropna=False)
        .rename_axis("status")
        .reset_index(name="count")
    )

    strong_cross = cross_candidates[
        cross_candidates["cross_source_status"]
        == "STRONG_DUPLICATE_CANDIDATE"
    ].sort_values(["date_difference_days", "distance_km"])

    print("\nStrong cross-source duplicate candidates:")
    display(strong_cross)

    print("\nClosest cross-source pairs for manual context:")
    display(
        cross_candidates.sort_values(
            ["distance_km", "date_difference_days"],
            na_position="last"
        ).head(30)
    )
else:
    strong_cross = pd.DataFrame()
    print("No JMG ↔ COOLR pairs found within 5 km.")

cross_candidates.to_csv(
    OUTPUT_DIR / "jmg_coolr_cross_source_candidates_within_5km.csv",
    index=False
)

strong_cross.to_csv(
    OUTPUT_DIR / "jmg_coolr_strong_duplicate_candidates.csv",
    index=False
)


Cross-source pairs within 5 km: 324


,status,count
0,REVIEW_ONLY,324



Strong cross-source duplicate candidates:


,jmg_record_id,jmg_source_event_id,jmg_date,jmg_state,jmg_district,jmg_location_1,jmg_location_2,jmg_latitude,jmg_longitude,coolr_record_id,...,coolr_location_accuracy,coolr_latitude,coolr_longitude,distance_km,date_difference_days,date_within_1_day,date_within_7_days,date_within_30_days,date_within_90_days,cross_source_status



Closest cross-source pairs for manual context:


,jmg_record_id,jmg_source_event_id,jmg_date,jmg_state,jmg_district,jmg_location_1,jmg_location_2,jmg_latitude,jmg_longitude,coolr_record_id,...,coolr_location_accuracy,coolr_latitude,coolr_longitude,distance_km,date_difference_days,date_within_1_day,date_within_7_days,date_within_30_days,date_within_90_days,cross_source_status
271,JMG_043,AY44201_M_02_SF_01,2021-01-06,Pahang,Cameron Highlands,Jalan Turun Bukit Fraser (KM 6.4),NaN,4.491990,101.385995,COOLR_5629,...,1km,4.492700,101.384200,0.2140,2614.0,False,False,False,False,REVIEW_ONLY
264,JMG_042,AY44201_M_01_SF_01,2021-05-29,Pahang,Cameron Highlands,Taman Gambang Damai,NaN,4.491990,101.385995,COOLR_5629,...,1km,4.492700,101.384200,0.2140,2757.0,False,False,False,False,REVIEW_ONLY
191,JMG_029,AY44103_LS_01,2021-12-19,Pahang,Cameron Highlands,Taman Jasmine,NaN,4.414110,101.380162,COOLR_6351,...,5km,4.411900,101.380100,0.2458,2600.0,False,False,False,False,REVIEW_ONLY
136,JMG_019,AU3103_N_06_L_01,2021-11-18,Selangor,Gombak,NaN,"Jalan Bukit Jaya 6a, Taman Bukit Jaya",3.185008,101.771559,COOLR_915,...,5km,3.186900,101.773000,0.2643,4730.0,False,False,False,False,REVIEW_ONLY
200,JMG_033,AY44201_LS_01,2021-12-19,Pahang,Cameron Highlands,NaN,jalan tengkolok,4.483359,101.373026,COOLR_11033,...,exact,4.485386,101.374293,0.2656,1820.0,False,False,False,False,REVIEW_ONLY
250,JMG_040,AY44201_LS_08,2021-12-19,Pahang,Cameron Highlands,Jalan Istana,NaN,4.492671,101.386642,COOLR_5629,...,1km,4.492700,101.384200,0.2707,2961.0,False,False,False,False,REVIEW_ONLY
141,JMG_020,AU3103_N_07_L_01,2021-12-18,Selangor,Gombak,NaN,Jalan Wangsa 1,3.185896,101.764254,COOLR_1173,...,5km,3.185700,101.767300,0.3389,4473.0,False,False,False,False,REVIEW_ONLY
142,JMG_020,AU3103_N_07_L_01,2021-12-18,Selangor,Gombak,NaN,Jalan Wangsa 1,3.185896,101.764254,COOLR_507,...,5km,3.185000,101.767300,0.3526,4987.0,False,False,False,False,REVIEW_ONLY
306,JMG_047,FH2101_M_01_SF_02,2021-09-28,Pulau Pinang,Timor Laut,NaN,Jalan Batu Ferigghi,5.467109,100.277564,COOLR_13237,...,exact,5.467079,100.274239,0.3681,1423.0,False,False,False,False,REVIEW_ONLY
303,JMG_047,FH2101_M_01_SF_02,2021-09-28,Pulau Pinang,Timor Laut,NaN,Jalan Batu Ferigghi,5.467109,100.277564,COOLR_5494,...,1km,5.466100,100.274300,0.3784,2939.0,False,False,False,False,REVIEW_ONLY


## 8. Apply only strong cross-source duplicates

If a strong JMG ↔ COOLR pair exists, the notebook keeps the JMG record as the representative because it is the official government inventory source for that matched event.

However, this happens **only** when the strong cross-source rule is satisfied.

If no strong cross-source candidates exist, nothing is removed.


In [11]:
cross_group_counter = 1

if len(strong_cross):
    # A COOLR event may theoretically match more than one JMG row.
    # Resolve only unambiguous one-to-one strong candidate pairs automatically.
    jmg_counts = strong_cross["jmg_record_id"].value_counts()
    coolr_counts = strong_cross["coolr_record_id"].value_counts()

    auto_pairs = strong_cross[
        strong_cross["jmg_record_id"].map(jmg_counts).eq(1)
        & strong_cross["coolr_record_id"].map(coolr_counts).eq(1)
    ].copy()

    ambiguous_strong = strong_cross.drop(auto_pairs.index)

    if len(ambiguous_strong):
        print(
            "WARNING: Some strong cross-source candidates are not one-to-one. "
            "They remain for manual review and will not be removed automatically."
        )
        display(ambiguous_strong)

    for _, pair in auto_pairs.iterrows():
        jmg_id = pair["jmg_record_id"]
        coolr_id = pair["coolr_record_id"]
        group_id = f"CROSS_DUP_{cross_group_counter:03d}"

        jmg_mask = jmg_work["source_record_id"] == jmg_id
        coolr_mask = coolr_work["source_record_id"] == coolr_id

        # JMG retained as representative.
        jmg_work.loc[jmg_mask, "duplicate_group"] = group_id
        jmg_work.loc[jmg_mask, "representative_record_id"] = jmg_id
        jmg_work.loc[jmg_mask, "duplicate_reason"] = (
            "Strong JMG-COOLR match: <=1 km and event dates within 7 days"
        )

        coolr_work.loc[coolr_mask, "duplicate_group"] = group_id
        coolr_work.loc[coolr_mask, "representative_record_id"] = jmg_id
        coolr_work.loc[coolr_mask, "duplicate_reason"] = (
            "Strong JMG-COOLR match: <=1 km and event dates within 7 days"
        )
        coolr_work.loc[coolr_mask, "dedup_status"] = "EXCLUDE_DUPLICATE"

        cross_group_counter += 1

    print("Automatically applied one-to-one cross-source duplicate groups:", len(auto_pairs))
else:
    print("No strong cross-source duplicates to apply.")


No strong cross-source duplicates to apply.


## 9. Build one standard positive inventory schema

The model does not need all source-specific columns in one table.

A standard schema is created while retaining source provenance.


In [12]:
# ------------------------------------------------------------
# COOLR standard table
# ------------------------------------------------------------
coolr_standard = gpd.GeoDataFrame({
    "positive_id": coolr_work["source_record_id"],
    "source_dataset": "NASA_COOLR",
    "source_record_id": coolr_work["source_record_id"],
    "source_event_id": coolr_work["event_id"].astype("string"),
    "event_date": coolr_work["event_date_standard"],
    "event_date_available": coolr_work["event_date_standard"].notna(),
    "event_date_source_reliable_for_review": coolr_work["event_date_standard"].notna(),
    "state_or_admin": coolr_work["admin_division_name"],
    "district_or_locality": coolr_work["location_description"],
    "event_title_or_name": coolr_work["event_title"],
    "location_accuracy_source": coolr_work["location_accuracy"],
    "longitude": coolr_work["final_longitude"],
    "latitude": coolr_work["final_latitude"],
    "dedup_status": coolr_work["dedup_status"],
    "duplicate_group": coolr_work["duplicate_group"],
    "representative_record_id": coolr_work["representative_record_id"],
    "duplicate_reason": coolr_work["duplicate_reason"],
    "geometry": coolr_work.geometry,
}, geometry="geometry", crs="EPSG:4326")

# ------------------------------------------------------------
# JMG standard table
# ------------------------------------------------------------
jmg_location = (
    jmg_work["Nama_Kampu"].astype("string").fillna("")
    + " | "
    + jmg_work["Nama_Jalan"].astype("string").fillna("")
)
jmg_location = (
    jmg_location
    .str.strip(" |")
    .replace("", pd.NA)
)

jmg_standard = gpd.GeoDataFrame({
    "positive_id": jmg_work["source_record_id"],
    "source_dataset": "JMG",
    "source_record_id": jmg_work["source_record_id"],
    "source_event_id": jmg_work["jmg_source_event_id"],
    "event_date": jmg_work["event_date_standard"],
    "event_date_available": jmg_work["event_date_standard"].notna(),

    # IMPORTANT:
    # This is only a date-availability flag for the inventory.
    # Rainfall reliability is assessed separately later.
    "event_date_source_reliable_for_review": jmg_work["source_date_complete_valid"].fillna(False),

    "state_or_admin": jmg_work["Negeri"],
    "district_or_locality": jmg_location,
    "event_title_or_name": jmg_work["Jenis_Tana"],
    "location_accuracy_source": pd.NA,
    "longitude": jmg_work["final_longitude"],
    "latitude": jmg_work["final_latitude"],
    "dedup_status": jmg_work["dedup_status"],
    "duplicate_group": jmg_work["duplicate_group"],
    "representative_record_id": jmg_work["representative_record_id"],
    "duplicate_reason": jmg_work["duplicate_reason"],
    "geometry": jmg_work.geometry,
}, geometry="geometry", crs="EPSG:4326")

combined_all = gpd.GeoDataFrame(
    pd.concat(
        [coolr_standard, jmg_standard],
        ignore_index=True
    ),
    geometry="geometry",
    crs="EPSG:4326"
)

print("Combined raw positive rows:", len(combined_all))
print(combined_all["source_dataset"].value_counts())


Combined raw positive rows: 288
source_dataset
NASA_COOLR    236
JMG            52
Name: count, dtype: int64


## 10. Final retained positive inventory

Only rows explicitly marked `EXCLUDE_DUPLICATE` are removed.

All uncertain/review-only cases remain retained.


In [13]:
final_positive = combined_all[
    combined_all["dedup_status"] == "KEEP"
].copy()

final_positive = final_positive.reset_index(drop=True)

# Stable final IDs for downstream model workflow.
final_positive.insert(
    0,
    "final_positive_id",
    [f"POS_{i:04d}" for i in range(1, len(final_positive) + 1)]
)

print("Raw candidates:", len(combined_all))
print("Excluded duplicate rows:", (combined_all["dedup_status"] == "EXCLUDE_DUPLICATE").sum())
print("Final retained positives:", len(final_positive))

print("\nRetained by source:")
display(
    final_positive["source_dataset"]
    .value_counts()
    .rename_axis("source_dataset")
    .reset_index(name="retained_records")
)


Raw candidates: 288
Excluded duplicate rows: 1
Final retained positives: 287

Retained by source:


,source_dataset,retained_records
0,NASA_COOLR,235
1,JMG,52


## 11. Final Malaysia-boundary validation


In [14]:
malaysia_for_check = malaysia.to_crs(final_positive.crs)

if len(malaysia_for_check) == 1:
    malaysia_geom = malaysia_for_check.geometry.iloc[0]
else:
    malaysia_geom = malaysia_for_check.dissolve().geometry.iloc[0]

final_positive["inside_malaysia_boundary"] = (
    final_positive.geometry.apply(malaysia_geom.covers)
)

display(
    final_positive["inside_malaysia_boundary"]
    .value_counts(dropna=False)
    .rename_axis("inside_malaysia")
    .reset_index(name="count")
)

if not final_positive["inside_malaysia_boundary"].all():
    raise ValueError(
        "At least one final positive record is outside the supplied Malaysia boundary."
    )

print("All final positive records are inside/on the supplied Malaysia boundary.")


,inside_malaysia,count
0,True,287


All final positive records are inside/on the supplied Malaysia boundary.


## 12. Spatial coverage summary

This is not yet proof of nationwide representativeness.  
It simply documents where the occurrence evidence is concentrated.


In [15]:
coverage_by_source = (
    final_positive
    .groupby("source_dataset")
    .agg(
        records=("final_positive_id", "size"),
        min_lon=("longitude", "min"),
        max_lon=("longitude", "max"),
        min_lat=("latitude", "min"),
        max_lat=("latitude", "max"),
        dated_records=("event_date_available", "sum"),
    )
    .reset_index()
)

display(coverage_by_source)

coverage_by_admin = (
    final_positive["state_or_admin"]
    .fillna("MISSING")
    .value_counts()
    .rename_axis("state_or_admin")
    .reset_index(name="positive_records")
)

display(coverage_by_admin)

coverage_by_source.to_csv(
    OUTPUT_DIR / "positive_coverage_by_source.csv",
    index=False
)

coverage_by_admin.to_csv(
    OUTPUT_DIR / "positive_coverage_by_admin.csv",
    index=False
)


,source_dataset,records,min_lon,max_lon,min_lat,max_lat,dated_records
0,JMG,52,100.277564,116.598097,3.100022,6.015133,47
1,NASA_COOLR,235,100.214496,118.404700,1.051748,6.412300,235


,state_or_admin,positive_records
0,Selangor,58
1,Sabah,51
2,Pahang,39
3,Pulau Pinang,37
4,Sarawak,35
5,Wilayah Persekutuan Kuala Lumpur,30
6,Perak,12
7,Kelantan,6
8,Negeri Sembilan,4
9,Terengganu,4


## 13. Map final positive inventory

The figure is saved to disk even if VS Code cannot render PNG output inline.


In [16]:
malaysia_4326 = malaysia.to_crs("EPSG:4326")

fig, ax = plt.subplots(figsize=(12, 9))

malaysia_4326.boundary.plot(
    ax=ax,
    linewidth=0.8
)

for source_name, group in final_positive.groupby("source_dataset"):
    group.plot(
        ax=ax,
        markersize=25 if source_name == "NASA_COOLR" else 45,
        alpha=0.75,
        marker="o" if source_name == "NASA_COOLR" else "x",
        label=source_name
    )

ax.set_title(
    "Final Reviewed Positive Landslide Inventory - Malaysia"
)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
ax.grid(True, alpha=0.2)

plt.tight_layout()

map_path = OUTPUT_DIR / "final_positive_inventory_map.png"

fig.savefig(
    map_path,
    dpi=180,
    bbox_inches="tight"
)

plt.close(fig)

print("Map saved to:")
print(map_path)


Map saved to:
C:\Landslide_FYP\Landslide_FYP_Development\01_preprocessing\positive_inventory_deduplication_outputs\final_positive_inventory_map.png


## 14. Save complete deduplication audit

This preserves both retained and excluded records.


In [17]:
audit_csv = combined_all.drop(
    columns="geometry",
    errors="ignore"
).copy()

audit_csv["event_date"] = audit_csv["event_date"].astype("string")

audit_csv.to_csv(
    OUTPUT_DIR / "positive_inventory_complete_dedup_audit.csv",
    index=False
)

excluded_records = audit_csv[
    audit_csv["dedup_status"] == "EXCLUDE_DUPLICATE"
].copy()

excluded_records.to_csv(
    OUTPUT_DIR / "positive_inventory_excluded_duplicates.csv",
    index=False
)

print("Complete audit rows:", len(audit_csv))
print("Excluded rows:", len(excluded_records))


Complete audit rows: 288
Excluded rows: 1


## 15. Save final positive inventory

These are the files to use in later pseudo-absence and feature-extraction notebooks.


In [18]:
final_geojson_path = (
    CLEAN_DIR
    / "malaysia_positive_inventory_final.geojson"
)

final_csv_path = (
    CLEAN_DIR
    / "malaysia_positive_inventory_final.csv"
)

final_export = final_positive.copy()
final_export["event_date"] = (
    final_export["event_date"]
    .astype("string")
    .replace("<NA>", None)
)

final_export.to_file(
    final_geojson_path,
    driver="GeoJSON"
)

final_csv = final_positive.drop(
    columns="geometry",
    errors="ignore"
).copy()

final_csv["event_date"] = final_csv["event_date"].astype("string")

final_csv.to_csv(
    final_csv_path,
    index=False
)

print("Saved final GeoJSON:")
print(final_geojson_path)
print()
print("Saved final CSV:")
print(final_csv_path)


Saved final GeoJSON:
C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\malaysia_positive_inventory_final.geojson

Saved final CSV:
C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\malaysia_positive_inventory_final.csv


## 16. Final Stage-3 readiness report


In [19]:
summary = pd.DataFrame({
    "check": [
        "Raw COOLR records",
        "Raw resolved JMG records",
        "Combined raw candidates",
        "COOLR duplicate rows excluded",
        "JMG duplicate rows excluded",
        "Cross-source duplicate rows excluded",
        "Total duplicate rows excluded",
        "Final unique positive records",
        "Final COOLR positives",
        "Final JMG positives",
        "Final positives inside Malaysia",
        "Final positives with event date",
        "Final positives without event date",
        "Strong cross-source candidate pairs",
    ],
    "value": [
        len(coolr_work),
        len(jmg_work),
        len(combined_all),
        int(
            (
                (coolr_work["dedup_status"] == "EXCLUDE_DUPLICATE")
                & coolr_work["duplicate_group"].astype("string").str.startswith("COOLR_DUP_", na=False)
            ).sum()
        ),
        int(
            (
                (jmg_work["dedup_status"] == "EXCLUDE_DUPLICATE")
                & jmg_work["duplicate_group"].astype("string").str.startswith("JMG_DUP_", na=False)
            ).sum()
        ),
        int(
            (
                (coolr_work["dedup_status"] == "EXCLUDE_DUPLICATE")
                & coolr_work["duplicate_group"].astype("string").str.startswith("CROSS_DUP_", na=False)
            ).sum()
        ),
        int((combined_all["dedup_status"] == "EXCLUDE_DUPLICATE").sum()),
        len(final_positive),
        int((final_positive["source_dataset"] == "NASA_COOLR").sum()),
        int((final_positive["source_dataset"] == "JMG").sum()),
        int(final_positive["inside_malaysia_boundary"].sum()),
        int(final_positive["event_date_available"].sum()),
        int((~final_positive["event_date_available"]).sum()),
        len(strong_cross),
    ]
})

display(summary)

summary.to_csv(
    OUTPUT_DIR / "positive_inventory_deduplication_summary.csv",
    index=False
)

print("\nSTAGE-3 DECISION")
print("----------------")

if len(strong_cross) == 0:
    print(
        "No strong JMG↔COOLR cross-source duplicates were found "
        "under the conservative <=1 km and <=7-day rule."
    )
else:
    print(
        "Strong cross-source candidate(s) were detected. "
        "Review jmg_coolr_strong_duplicate_candidates.csv."
    )

print()
print("Final positive count:", len(final_positive))
print()
print(
    "If the excluded-record audit is reasonable, "
    "the positive inventory is ready for pseudo-absence design."
)
print()
print(
    "NEXT: 04_pseudo_absence_generation.ipynb"
)


,check,value
0,Raw COOLR records,236
1,Raw resolved JMG records,52
2,Combined raw candidates,288
3,COOLR duplicate rows excluded,1
4,JMG duplicate rows excluded,0
5,Cross-source duplicate rows excluded,0
6,Total duplicate rows excluded,1
7,Final unique positive records,287
8,Final COOLR positives,235
9,Final JMG positives,52



STAGE-3 DECISION
----------------
No strong JMG↔COOLR cross-source duplicates were found under the conservative <=1 km and <=7-day rule.

Final positive count: 287

If the excluded-record audit is reasonable, the positive inventory is ready for pseudo-absence design.

NEXT: 04_pseudo_absence_generation.ipynb


# Stage-3 output structure

After successful execution:

```text
Landslide_FYP_Development/
├── 01_preprocessing/
│   └── positive_inventory_deduplication_outputs/
│       ├── coolr_strong_duplicate_review.csv
│       ├── jmg_within_source_duplicate_review.csv
│       ├── jmg_coolr_cross_source_candidates_within_5km.csv
│       ├── jmg_coolr_strong_duplicate_candidates.csv
│       ├── positive_inventory_complete_dedup_audit.csv
│       ├── positive_inventory_excluded_duplicates.csv
│       ├── positive_inventory_deduplication_summary.csv
│       ├── positive_coverage_by_source.csv
│       ├── positive_coverage_by_admin.csv
│       └── final_positive_inventory_map.png
│
└── 02_cleaned_inventory/
    ├── malaysia_positive_inventory_final.geojson
    └── malaysia_positive_inventory_final.csv
```

## What comes next

Do **not** train models yet.

The next notebook should be:

### `04_pseudo_absence_generation.ipynb`

That stage will:

1. use the final reviewed positive inventory,
2. design exclusion buffers around positives,
3. create reproducible non-landslide candidate samples,
4. avoid open water and invalid pixels,
5. distribute pseudo-absences across Malaysia,
6. avoid excessive clustering,
7. compare 1:1, 1:2 and/or 1:3 sampling options,
8. save the selected positive/pseudo-absence sample set.

Feature extraction follows after the sampling design is validated.
